##MDP Fundamentals

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class S:
    def __init__(self, name, terminal=False):
        self.name = name
        self.terminal = terminal

    def __repr__(self):
        return self.name


In [ ]:
class MDP:
    def __init__(self):
        # Define States
        self.states = [
            S("s0"),
            S("s1"),
            S("s2"),
            S("s3", terminal=True)
        ]

        # Define Actions
        self.actions = ["a0", "a1"]

        self.n_states = len(self.states)
        self.n_actions = len(self.actions)

        # Transition Probabilities
        # Shape = (actions, states, next_states)
        self.P = np.zeros((self.n_actions, self.n_states, self.n_states))

        # ---------- Action a0 ----------
        self.P[0, 0] = [0.0, 1.0, 0.0, 0.0]      # s0 -> s1
        self.P[0, 1] = [0.0, 0.2, 0.8, 0.0]      # s1 -> stochastic
        self.P[0, 2] = [0.0, 0.0, 0.0, 1.0]      # s2 -> s3
        self.P[0, 3] = [0.0, 0.0, 0.0, 1.0]      # terminal stays

        # ---------- Action a1 ----------
        self.P[1, 0] = [0.0, 0.0, 1.0, 0.0]      # s0 -> s2
        self.P[1, 1] = [1.0, 0.0, 0.0, 0.0]      # s1 -> s0
        self.P[1, 2] = [0.0, 1.0, 0.0, 0.0]      # s2 -> s1
        self.P[1, 3] = [0.0, 0.0, 0.0, 1.0]      # terminal stays

        # Reward Matrix R(s,a)
        # Rows = states, Columns = actions
        self.R = np.array([
            [-1, -2],   # s0
            [ 5, -1],   # s1
            [10,  0],   # s2
            [ 0,  0]    # s3 (terminal)
        ])

    def print_transition_matrices(self):
        for a_idx, action in enumerate(self.actions):
            print(f"\nTransition Matrix for Action {action}")
            print("-" * 40)
            print(self.P[a_idx])

    def print_reward_matrix(self):
        print("\nReward Matrix R(s,a)")
        print("-" * 40)
        print("Rows    : States (s0,s1,s2,s3)")
        print("Columns : Actions (a0,a1)")
        print(self.R)

    def describe(self):
        print("\nEnvironment Description")
        print("-" * 40)
        print("States:")
        for s in self.states:
            if s.terminal:
                print(f"  {s} (Terminal)")
            else:
                print(f"  {s}")

        print("\nActions:")
        print("  a0")
        print("  a1")

        print("\nCharacteristics:")
        print("- Episodic MDP")
        print("- Terminal state: s3")
        print("- Stochastic transition:")
        print("    P(s2 | s1, a0) = 0.8")
        print("    P(s1 | s1, a0) = 0.2")
        print("- All transitions from s3 remain in s3.")



In [ ]:
if __name__ == "__main__":
    mdp = MDP()

    mdp.describe()
    mdp.print_transition_matrices()
    mdp.print_reward_matrix()

##Exploring nx module

In [ ]:
import networkx as nx

In [ ]:
G = nx.petersen_graph()

In [ ]:
G = nx.dodecahedral_graph()
options = {
    'node_color': 'black',
    'node_size': 100,
    'width': 3,
}
shells = [[2, 3, 4, 5, 6], [8, 1, 0, 19, 18, 17, 16, 15, 14, 7], [9, 10, 11, 12, 13]]
nx.draw_shell(G, nlist=shells, **options)
nx.draw(G)

##Visualizing the MDP

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

In [ ]:
def visualize(mdp):
    G = nx.MultiDiGraph()   # Allows multiple edges between same nodes

    # Add states as nodes
    for state in mdp.states:
        G.add_node(state.name)

    # Add transitions as edges
    for a_idx, action in enumerate(mdp.actions):
        for s in range(mdp.n_states):
            for sp in range(mdp.n_states):

                prob = mdp.P[a_idx, s, sp]

                if prob > 0:
                    reward = mdp.R[s, a_idx]

                    G.add_edge(
                        mdp.states[s].name,
                        mdp.states[sp].name,
                        action=action,
                        probability=prob,
                        reward=reward
                    )

    # Layout manual
    # pos = {
    #     "s0": (0, 0),
    #     "s1": (1, 1),
    #     "s2": (1, -1),
    #     "s3": (2, 0)
    # }
    

    # Layout using Module NetworkX
    pos = nx.kamada_kawai_layout(G)
    # pos = nx.shell_layout(G)
    # pos = nx.spectral_layout(G)
    # pos = nx.planar_layout(G)
    # pos = nx.circular_layout(G)
    # pos = nx.random_layout(G) #only for debugging
    # pos = graphviz_layout(G, prog="dot") #needs additional module avoid for small assignment

    plt.figure(figsize=(10,6))

    # Node colors
    node_colors = [
        "lightgreen" if state.terminal else "skyblue"
        for state in mdp.states
    ]

    nx.draw_networkx_nodes(
        G, pos,
        node_color=node_colors,
        node_size=1800
    )

    nx.draw_networkx_labels(
        G,
        pos,
        font_size=12,
        font_weight="bold"
    )

    # Separate edges by action
    a0_edges = []
    a1_edges = []


    for u, v, key, data in G.edges(keys=True, data=True):
        if data["action"] == "a0":
            a0_edges.append((u, v, key))
        else:
            a1_edges.append((u, v, key))
    # Draw edges
    # nx.draw_networkx_edges(
    #     G,
    #     pos,
    #     arrows=True,
    #     arrowstyle='-|>',
    #     arrowsize=20,
    #     connectionstyle="arc3,rad=0.15"
    # )
    # Draw a0 edges
    nx.draw_networkx_edges(
        G,
        pos,
        edgelist=a0_edges,
        edge_color="blue",
        arrows=True,
        arrowstyle='-|>',
        arrowsize=20,
        width=2,
        connectionstyle="arc3,rad=0.15"
    )


    # Draw a1 edges
    nx.draw_networkx_edges(
        G,
        pos,
        edgelist=a1_edges,
        edge_color="darkred",
        style="dashed",
        arrows=True,
        arrowstyle='-|>',
        arrowsize=20,
        width=2,
        connectionstyle="arc3,rad=-0.15"
    )

    
    # Edge labels
    edge_labels = {}
    a0_labels = {}
    a1_labels = {}
    for u, v, key, data in G.edges(keys=True, data=True):
        
            label = f"P={data['probability']:.1f}\nR={data['reward']}"
        
            if data["action"] == "a0":
                a0_labels[(u, v, key)] = label
            else:
                a1_labels[(u, v, key)] = label
    

    # for u, v, key, data in G.edges(keys=True, data=True):
    #     edge_labels[(u, v, key)] = (
    #         f"{data['action']}\n"
    #         f"P={data['probability']:.1f}\n"
    #         f"R={data['reward']}"
    #     )

    # nx.draw_networkx_edge_labels(
    #     G,
    #     pos,
    #     edge_labels=edge_labels,
    #     font_size=8
    # )
    nx.draw_networkx_edge_labels(
    G,
    pos,
    edge_labels=a0_labels,
    font_color="blue",
    font_size=8,
    rotate=False
)

    nx.draw_networkx_edge_labels(
    G,
    pos,
    edge_labels=a1_labels,
    font_color="darkred",
    font_size=8,
    rotate=False
    )

    # Legend
    legend_elements = [

        # Node legends
        Patch(
            facecolor="skyblue",
            edgecolor="black",
            label="Normal State"
        ),

        Patch(
            facecolor="lightgreen",
            edgecolor="black",
            label="Terminal State"
        ),

        # Action legends
        Line2D(
            [0],
            [0],
            color="blue",
            lw=2,
            label="Action a0"
        ),

        Line2D(
            [0],
            [0],
            color="darkred",
            lw=2,
            linestyle="--",
            label="Action a1"
        )
    ]

    plt.legend(
        handles=legend_elements,
        title="MDP Legend",
        loc="upper left",
        fontsize=9
    )

    
    plt.title("4-State Episodic MDP")
    plt.axis("off")
    plt.show()

In [ ]:
mdp = MDP()

mdp.describe()
mdp.print_transition_matrices()
mdp.print_reward_matrix()

visualize(mdp)

##Simulating Episodes

In [89]:
def random_policy(mdp, state):
    return np.random.choice(mdp.n_actions)

In [90]:

def simulate_episode(mdp, max_length=10):
    
    trajectory = []
    total_return = 0

    # Start from s0
    current_state = 0

    for step in range(max_length):

        # Stop if terminal state reached
        if mdp.states[current_state].terminal:
            print("Reached terminal state:",mdp.states[current_state].name)
            break

        # Random policy: choose action randomly
        # action_idx = np.random.choice(mdp.n_actions)
        action_idx = random_policy(mdp, current_state)
        action = mdp.actions[action_idx]

        # Get reward R(s,a)
        reward = int(mdp.R[current_state, action_idx])

        # Store (state, action, reward)
        trajectory.append(
            (
                mdp.states[current_state].name,
                action,
                reward
            )
        )

        total_return += reward

        # Sample next state using transition probabilities
        next_state = np.random.choice(
            mdp.n_states,
            p=mdp.P[action_idx, current_state]
        )

        current_state = next_state


    return trajectory, total_return

In [91]:
def simulate_episodes(mdp, num_episodes=10, max_length=10):

    returns = []

    for episode in range(num_episodes):

        trajectory, total_return = simulate_episode(
            mdp,
            max_length
        )

        returns.append(total_return)

        print(f"\nEpisode {episode+1}")
        print("-"*30)

        for step in trajectory:
            print(step)

        print("Return =", total_return)


    average_return = np.mean(returns)

    print("\n==============================")
    print("Average Return over episodes:", average_return)
    print("==============================")


    return returns

In [92]:
mdp = MDP()

returns = simulate_episodes(
    mdp,
    num_episodes=10,
    max_length=10
)


Episode 1
------------------------------
('s0', 'a0', -1)
('s1', 'a1', -1)
('s0', 'a0', -1)
('s1', 'a1', -1)
('s0', 'a1', -2)
('s2', 'a1', 0)
('s1', 'a1', -1)
('s0', 'a0', -1)
('s1', 'a1', -1)
('s0', 'a0', -1)
Return = -10
Reached terminal state: s3

Episode 2
------------------------------
('s0', 'a1', -2)
('s2', 'a0', 10)
Return = 8
Reached terminal state: s3

Episode 3
------------------------------
('s0', 'a1', -2)
('s2', 'a0', 10)
Return = 8
Reached terminal state: s3

Episode 4
------------------------------
('s0', 'a0', -1)
('s1', 'a0', 5)
('s2', 'a0', 10)
Return = 14
Reached terminal state: s3

Episode 5
------------------------------
('s0', 'a1', -2)
('s2', 'a0', 10)
Return = 8
Reached terminal state: s3

Episode 6
------------------------------
('s0', 'a0', -1)
('s1', 'a1', -1)
('s0', 'a0', -1)
('s1', 'a1', -1)
('s0', 'a1', -2)
('s2', 'a0', 10)
Return = 4
Reached terminal state: s3

Episode 7
------------------------------
('s0', 'a0', -1)
('s1', 'a1', -1)
('s0', 'a1', -2)
(